In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DecimalType, TimestampType, ArrayType)

dbutils.widgets.text("config", "", "Config record (JSON)")

raw_config=dbutils.widgets.get("config")
REQUIRED_KEYS = ["source_name", "source_folder", "ingestion_method",
                 "is_active", "target_schema", "target_table","target_catalog"]
def parse_config(raw: str) -> dict:
    """Parse the JSON config passed by the for-each task (object, or a single-item list)."""
    raw = (raw or "").strip()
    if not raw:
        raise ValueError("Parameter 'config' is empty. Pass one config record as JSON (for-each: {{input}}).")
    cfg = json.loads(raw)
    if isinstance(cfg, list):
        if len(cfg) != 1:
            raise ValueError(f"Expected one config record per run, got a list of {len(cfg)}. "
                             "Use this notebook inside a for-each task so each iteration gets one record.")
        cfg = cfg[0]
    missing = [k for k in REQUIRED_KEYS if cfg.get(k) in (None, "")]
    if missing:
        raise ValueError(f"Config is missing required keys: {missing}")

    # Normalise values
    cfg["ingestion_method"] = str(cfg["ingestion_method"]).strip().lower()
    cfg["source_format"] = str(cfg["source_format"]).strip().lower()
    cfg["load_type"] = str(cfg["load_type"]).strip().lower()
    cfg["is_active"] = cfg["is_active"] if isinstance(cfg["is_active"], bool) \
        else str(cfg["is_active"]).strip().lower() in ("true", "1", "yes")

    if cfg["source_format"] not in ("csv", "tsv", "json", "parquet"):
        raise ValueError(f"Unsupported source_format: {cfg['source_format']}")
    if cfg["load_type"] not in ("incremental", "overwrite"):
        raise ValueError(f"Unsupported load_type: {cfg['load_type']}")
    return cfg


In [0]:
cfg = parse_config(raw_config)
catalog = cfg["target_catalog"]
topic = cfg["source_folder"]
target_schema=cfg['target_schema']
target_table_load=cfg['target_table']
print(cfg)

In [0]:
def create_kafka_options(catalog,k_server_key,k_api_key,k_api_secret,k_topic):

    kafka_bootstrap_server = dbutils.secrets.get(catalog=catalog, schema="configs", key=k_server_key)

    api_key    =dbutils.secrets.get(catalog=catalog, schema="configs", key=k_api_key)

    api_secret =dbutils.secrets.get(catalog=catalog, schema="configs", key=k_api_secret)

    kafka_options = {
        "kafka.bootstrap.servers": kafka_bootstrap_server,
        "kafka.security.protocol": "SASL_SSL",
        "kafka.sasl.mechanism": "PLAIN",
        "kafka.sasl.jaas.config": (
            "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
            f'username="{api_key}" password="{api_secret}";'
        ),
        "kafka.ssl.endpoint.identification.algorithm": "https",
        "subscribe": k_topic,
        "startingOffsets": "earliest",
    }
    return kafka_options

In [0]:
kafka_options=create_kafka_options(catalog,"kafka_bootstrap_server","kafka_api_key","kafka_api_secret",topic)
# kafka_options=create_kafka_options("dev","kafka_bootstrap_server","kafka_api_key","kafka_api_secret","salesorderdetail")

value_format      = "json"                   
target_table = f"`{catalog}`.`{target_schema}`.`{target_table_load}`"
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{target_schema}`.`checkpoints`")

checkpoint_path = f"/Volumes/{catalog}/{target_schema}/checkpoints/kafka/{table}"
print(json.dumps(cfg, indent=2, default=str))
print("Target table   :", target_table)
print("topic:", topic)

In [0]:
VALUE_STR = "trim(cast(CASE WHEN substring(value, 1, 1) = X'00' THEN substring(value, 6) ELSE value END AS string))"

sample_opts = {k: v for k, v in kafka_options.items() if k != "startingOffsets"}
value_schema = (spark.read.format("kafka").options(**sample_opts)
                .option("startingOffsets", "earliest").option("endingOffsets", "latest").load()
                .selectExpr(f"{VALUE_STR} AS value_str").limit(1000)
                .selectExpr("schema_of_json_agg(value_str)").first()[0])
print("Inferred value schema:", value_schema)

# 2) Stream: parse the value with that schema and flatten every key into its own column
raw = (spark.readStream.format("kafka").options(**kafka_options)
       .option("failOnDataLoss", "false")          # trial topics have short retention
       .load())

parsed = (raw
    .withColumn("value_str", F.expr(VALUE_STR))
    .withColumn("data", F.from_json("value_str", value_schema))
    .select(
        "data.*",                                              # one column per JSON key
        F.col("key").cast("string").alias("_kafka_key"),
        F.col("topic").alias("_kafka_topic"),
        F.col("partition").alias("_kafka_partition"),
        F.col("offset").alias("_kafka_offset"),
        F.col("timestamp").alias("_kafka_timestamp"),
        F.col("value_str").alias("_raw_value"),                # kept for troubleshooting
        F.current_timestamp().alias("_ingested_at"),
    ))

query = (parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")       # new JSON keys in later runs are added as columns
    .trigger(availableNow=True)          # process everything available, then stop
    .toTable(target_table))
query.awaitTermination()

print("ingestion completed")

